In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Forecasting/master_long_hourly_test_2014_05_12.csv" "/content/master_long_hourly_test_2014_05_12.csv"
!cp "/content/drive/MyDrive/Forecasting/master_long_hourly_train_2012_2013.csv" "/content/master_long_hourly_train_2012_2013.csv"
!cp "/content/drive/MyDrive/Forecasting/master_long_hourly_validation_2014_01_04.csv" "/content/master_long_hourly_validation_2014_01_04.csv"
!cp "/content/drive/MyDrive/Forecasting/calendar_features_hourly.csv" "/content/calendar_features_hourly.csv"

In [3]:
!pip install statsforecast

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 212.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [4]:
from pathlib import Path

# Input files — produced by preprocessing + data-split teammate
TRAIN_PATH    = "/content/master_long_hourly_train_2012_2013.csv"
VAL_PATH      = "/content/master_long_hourly_validation_2014_01_04.csv"
TEST_PATH     = "/content/master_long_hourly_test_2014_05_12.csv"
CALENDAR_PATH = "/content/calendar_features_hourly.csv"

# Output — share this file with teammates for alignment
SELECTED_CLIENTS_PATH = Path("selected_clients_sarimax.csv")

# Sampling
RANDOM_SEED = 42
# N_CLIENTS   = 50    # increase carefully — SARIMAX is slow per client

# Model
SEASON_LENGTH = 24  # hourly data → daily seasonality
LOOKBACK_HOURS = 672 # 4 weeks lookback
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

In [5]:
# IMPORTS
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import ARIMA, AutoARIMA

In [6]:
# ── SAMPLE CLIENTS ───────────────────────────────────────────────────────────
# Read only the client_id column to avoid loading the full file
train_ids = pd.read_csv(TRAIN_PATH, usecols=["client_id"])
all_clients = train_ids["client_id"].unique()

# rng = np.random.default_rng(RANDOM_SEED)
# selected = sorted(rng.choice(all_clients, size=N_CLIENTS, replace=False).tolist())
selected = sorted(all_clients.tolist()) # Use all clients

print(f"Selected {len(selected)} clients (ALL)")

# Save so teammates can use the same clients
pd.DataFrame({"client_id": selected}).to_csv(SELECTED_CLIENTS_PATH, index=False)
print(f"Saved client list → {SELECTED_CLIENTS_PATH}")

Selected 156 clients (ALL)
Saved client list → selected_clients_sarimax.csv


In [7]:
# ── PREPARE DATA ─────────────────────────────────────────────────────────────
# Cyclic encodings (sin/cos) capture periodicity without ordinality.
# is_weekend adds a structural break between weekdays and weekends.
# Skipping raw integer columns (hour, day_of_week, etc.) — cyclic versions are better for SARIMAX.
EXOG_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend", "month_sin", "month_cos"]

calendar = pd.read_csv(CALENDAR_PATH, parse_dates=["timestamp"])


def load_split(path, clients, calendar, exog_cols):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    df = df[df["client_id"].isin(clients)].copy()
    df = df.merge(calendar[["timestamp"] + exog_cols], on="timestamp", how="left")
    # StatsForecast expects: unique_id, ds, y
    df = df.rename(columns={"client_id": "unique_id", "timestamp": "ds"})
    return df.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def apply_lookback(df, hours):
    """Keep only the last `hours` rows per client."""
    return (df.groupby("unique_id", group_keys=False)
              .tail(hours)
              .reset_index(drop=True))


train = load_split(TRAIN_PATH, selected, calendar, EXOG_COLS)
val   = load_split(VAL_PATH,   selected, calendar, EXOG_COLS)
test  = load_split(TEST_PATH,  selected, calendar, EXOG_COLS)

# Apply 4-week lookback: train on the most recent 672 hours per client
train = apply_lookback(train, LOOKBACK_HOURS)

print(f"train : {train.shape}  {train['ds'].min()} → {train['ds'].max()}")
print(f"val   : {val.shape}    {val['ds'].min()} → {val['ds'].max()}")
print(f"test  : {test.shape}   {test['ds'].min()} → {test['ds'].max()}")

train : (104832, 10)  2013-12-04 00:00:00 → 2013-12-31 23:00:00
val   : (445536, 10)    2014-01-01 00:00:00 → 2014-04-30 23:00:00
test  : (913536, 10)   2014-05-01 00:00:00 → 2014-12-31 23:00:00


In [8]:
# ── EVALUATION METRICS ────────────────────────────────────────────────────────
def compute_metrics(df, target_col="y", pred_col="ARIMA"):
    y    = df[target_col].values
    yhat = df[pred_col].values
    mse  = np.mean((y - yhat) ** 2)
    mae  = np.mean(np.abs(y - yhat))
    wape = np.sum(np.abs(y - yhat)) / np.sum(np.abs(y))
    return {"MSE": round(mse, 4), "MAE": round(mae, 4), "WAPE": round(wape, 4)}


def per_client_metrics(df, target_col="y", pred_col="ARIMA"):
    rows = []
    for uid, grp in df.groupby("unique_id"):
        rows.append({"client_id": uid, **compute_metrics(grp, target_col, pred_col)})
    rows.append({"client_id": "OVERALL", **compute_metrics(df, target_col, pred_col)})
    return pd.DataFrame(rows)

In [ ]:
# ── TRAIN SARIMAX ─────────────────────────────────────────────────────────────
# Switching to AutoARIMA allows the model to select the best parameters (p,d,q)
# automatically for each client, preventing the poor fit of a fixed order.
model = AutoARIMA(
    season_length=SEASON_LENGTH,
    approximation=True,  # Faster search
    stepwise=True        # Faster search
)

sf = StatsForecast(models=[model], freq="h", n_jobs=-1)
sf.fit(train[["unique_id", "ds", "y"] + EXOG_COLS])

joblib.dump(sf, MODEL_DIR / "sarimax_val.joblib")
print("Training complete. Model saved → models/sarimax_val.joblib")

Training complete. Model saved → models/sarimax_val.joblib


In [ ]:
# ── VALIDATION FORECAST ───────────────────────────────────────────────────────
# h = number of unique timestamps in val (same for all clients after a clean time-based split)
val_h = val["ds"].nunique()
X_val = val[["unique_id", "ds"] + EXOG_COLS]

val_preds = sf.predict(h=val_h, X_df=X_val)
val_eval  = val[["unique_id", "ds", "y"]].merge(val_preds, on=["unique_id", "ds"])

print(val_eval.head())

  unique_id                  ds          y   AutoARIMA
0    MT_124 2014-01-01 00:00:00  95.693780  132.197285
1    MT_124 2014-01-01 01:00:00  83.732056  122.075237
2    MT_124 2014-01-01 02:00:00  82.535890  106.343234
3    MT_124 2014-01-01 03:00:00  82.535890   71.045445
4    MT_124 2014-01-01 04:00:00  63.397133   31.193867


In [ ]:
# 1. Training Metrics (In-Sample)
# Access fitted models directly and call predict_in_sample()
train_preds_list = []
for i, uid in enumerate(train["unique_id"].unique()):
    fitted_model = sf.fitted_[i, 0]  # [series_index, model_index]
    client_train = train[train["unique_id"] == uid].copy()

    # Get in-sample predictions
    in_sample_preds = fitted_model.predict_in_sample()

    # Check if it's a dict and extract the mean, otherwise use directly
    if isinstance(in_sample_preds, dict):
        preds = in_sample_preds.get("mean", in_sample_preds.get("fitted", list(in_sample_preds.values())[0]))
    else:
        preds = in_sample_preds

    # Create dataframe with predictions
    pred_df = pd.DataFrame({
        "unique_id": [uid] * len(client_train),
        "ds": client_train["ds"].values,
        "AutoARIMA": preds
    })
    train_preds_list.append(pred_df)

train_preds = pd.concat(train_preds_list, ignore_index=True)
train_eval = train[["unique_id", "ds", "y"]].merge(train_preds, on=["unique_id", "ds"], how="left")

train_metrics = per_client_metrics(train_eval.dropna(), pred_col="AutoARIMA")
print("── Training Metrics (In-Sample) ──")
# Show first 5 and OVERALL
print(train_metrics.head(5).to_string(index=False))
print("...")
print(train_metrics.tail(1).to_string(index=False))
print("\n")

# 2. Validation Metrics (Out-of-Sample)
val_metrics = per_client_metrics(val_eval, pred_col="AutoARIMA")
print("── Validation Metrics (Out-of-Sample) ──")
# Show first 5 and OVERALL
print(val_metrics.head(5).to_string(index=False))
print("...")
print(val_metrics.tail(1).to_string(index=False))

── Training Metrics (In-Sample) ──
client_id      MSE     MAE   WAPE
   MT_124 918.4121 21.8408 0.0816
   MT_132  69.6918  4.8397 0.2979
   MT_156  82.3269  6.3680 0.0948
   MT_158 326.4842 12.4478 0.1673
   MT_159 176.1831  8.1321 0.1325
...
client_id        MSE     MAE   WAPE
  OVERALL 14888.6629 27.0645 0.0435


── Validation Metrics (Out-of-Sample) ──
client_id        MSE      MAE   WAPE
   MT_124 10790.5017  82.5459 0.3049
   MT_132   179.5037  11.0396 0.6456
   MT_156  1833.2875  36.2980 0.4446
   MT_158 23073.7712 144.8790 1.9635
   MT_159  3001.2661  46.8198 0.9196
...
client_id         MSE    MAE   WAPE
  OVERALL 396656.4188 184.28 0.3004


In [9]:
# ── TEST EVALUATION ───────────────────────────────────────────────────────────
# Retrain on train+val (with lookback), then predict on the held-out test set
train_val = pd.concat([train, val], ignore_index=True).sort_values(["unique_id", "ds"])
train_val = apply_lookback(train_val, LOOKBACK_HOURS)

# Redefine the model here to ensure it's available
model = AutoARIMA(
    season_length=SEASON_LENGTH,
    approximation=True,  # Faster search
    stepwise=True        # Faster search
)

sf_final = StatsForecast(models=[model], freq="h", n_jobs=-1)
sf_final.fit(train_val[["unique_id", "ds", "y"] + EXOG_COLS])

joblib.dump(sf_final, MODEL_DIR / "sarimax_final.joblib")
print("Final model saved → models/sarimax_final.joblib")

test_h     = test["ds"].nunique()
X_test     = test[["unique_id", "ds"] + EXOG_COLS]
test_preds = sf_final.predict(h=test_h, X_df=X_test)
test_eval  = test[["unique_id", "ds", "y"]].merge(test_preds, on=["unique_id", "ds"])

test_metrics = per_client_metrics(test_eval, pred_col="AutoARIMA")
print("── Test metrics ──")
print(test_metrics.to_string(index=False))

Final model saved → models/sarimax_final.joblib
── Test metrics ──
client_id          MSE       MAE   WAPE
   MT_124 3.500138e+03   43.3344 0.1443
   MT_132 8.258370e+02   23.4156 1.5472
   MT_156 7.060933e+02   19.2063 0.2159
   MT_158 1.598990e+03   32.8845 0.4196
   MT_159 2.423343e+03   36.2883 0.8475
   MT_161 4.907087e+06 1993.1125 1.2711
   MT_162 2.437992e+04  125.5413 0.4631
   MT_163 1.241058e+07 3215.3950 1.2859
   MT_166 2.728217e+05  416.6519 0.3225
   MT_168 1.667131e+02    9.8795 0.0738
   MT_169 3.557310e+01    4.9631 0.1178
   MT_171 5.293966e+03   57.7759 0.2446
   MT_172 3.407614e+02   13.7668 0.1126
   MT_174 8.012261e+02   19.2427 0.1171
   MT_175 4.938399e+02   15.5307 0.0977
   MT_176 1.994614e+02   10.4205 0.1208
   MT_180 7.837532e+02   20.8932 0.1398
   MT_182 4.835100e+01    5.0530 0.1004
   MT_183 1.679116e+02    9.2784 0.1198
   MT_187 3.388479e+02   14.2462 0.1533
   MT_188 1.639847e+02    9.2560 0.2183
   MT_189 4.098970e+03   46.7048 0.1167
   MT_190 2.4